# Fitting a GP with `tinygp`
From [this](https://tinygp.readthedocs.io/en/latest/tutorials/means.html), fitting the mean of a GP.

As the mean is non-linear (and we also infer the hyperparameters of the GP kernel), closed form not possible and the optimisation is performed using `optax` first order methods.

In [ ]:
from functools import partial
import matplotlib.pyplot as plt
import numpy as np
import jax

jax.config.update("jax_enable_x64", True)

import jaxopt
import jax.numpy as jnp
from tinygp import GaussianProcess, kernels


def mean_function(params, X):
    mod = jnp.exp(-0.5 * jnp.square((X - params["loc"]) / jnp.exp(params["log_width"])))
    beta = jnp.array([1, mod])
    return params["amps"] @ beta


mean_params = {
    "amps": np.array([0.1, 0.3]),
    "loc": 5.0,
    "log_width": np.log(0.5),
}
"""

def mean_function(params, X):
    mod = jax.nn.sigmoid((X - params["tau"]) / params["omega"])
    beta = jnp.array([1, mod])
    return params["amps"] @ beta

mean_params = {
    "amps": np.array([0, 0.5]),  # with 0 we neglect the y intercept
    #"amps": np.array([0.3]),
    "omega": 0.5,
    "tau": 7.2,
}
"""
X_grid = np.linspace(0, 10, 200)
model = jax.vmap(partial(mean_function, mean_params))(X_grid)

plt.plot(X_grid, model)
plt.xlabel("x")
plt.ylabel("y")
_ = plt.title("a parametric mean model")

In [ ]:
# simulate data
random = np.random.default_rng(135)
X = np.sort(random.uniform(0, 10, 50))
y = jax.vmap(partial(mean_function, mean_params))(X)
# background trend that’s not included in the mean model
y += 0.1 * np.sin(2 * np.pi * (X - 5) / 10.0)
y += 0.03 * random.normal(size=len(X))
# plus some noise
plt.plot(X, y, ".k")
plt.xlabel("x")
plt.ylabel("y")
_ = plt.title("simulated data")

In [ ]:
vmapped_mean_function = jax.vmap(mean_function, in_axes=(None, 0))


# remember that with new tinygp, v2 in "Alternative approach" is
# the only way to fit the mean, the other approach doesn't work
# see https://github.com/dfm/tinygp/issues/261
def build_gp_v2(params):
    kernel = jnp.exp(params["log_gp_amp"]) * kernels.Matern52(
        jnp.exp(params["log_gp_scale"])
    )
    return GaussianProcess(kernel, X, diag=jnp.exp(params["log_gp_diag"]))


@jax.jit
def loss_v2(params):
    gp = build_gp_v2(params)
    return -gp.log_probability(y - vmapped_mean_function(params, X))


params = dict(
    log_gp_amp=np.log(0.1),
    log_gp_scale=np.log(3.0),
    log_gp_diag=np.log(0.03),
    **mean_params,
)
print(f"Initial negative log likelihood: {loss_v2(params)}")
solver = jaxopt.ScipyMinimize(fun=loss_v2)
soln = solver.run(jax.tree_util.tree_map(jnp.asarray, params))
print(f"Final negative log likelihood: {soln.state.fun_val}")

In [ ]:
gp = build_gp_v2(soln.params)

# rm mean from data
_, cond = gp.condition(y - vmapped_mean_function(params, X), X_grid, include_mean=False)
_, cond_no_mean = gp.condition(
    y - vmapped_mean_function(params, X), X_grid, include_mean=False
)
# remember to add mean back
mu = cond.loc + vmapped_mean_function(params, X_grid)
std = np.sqrt(cond.variance)
mu_no_mean = cond_no_mean.loc + soln.params["amps"][0]
std_no_mean = np.sqrt(cond_no_mean.variance)

fig, axes = plt.subplots(
    1, 2, figsize=(6, 3), layout="constrained", sharex=True, sharey=True
)
ax0, ax1 = axes[0], axes[1]
ax0.plot(X, y, ".k", label="data")
ax0.plot(X_grid, mu, label="model")
ax0.fill_between(X_grid, mu + std, mu - std, color="C0", alpha=0.3)

ax0.set_xlim(X_grid.min(), X_grid.max())
ax0.set_xlabel("x")
ax0.set_ylabel("y")
ax0.legend().set_visible(True)
ax1.plot(X, y, ".k", label="data")
ax1.plot(X_grid, mu_no_mean, label="GP model")
ax1.fill_between(
    X_grid, mu_no_mean + std_no_mean, mu_no_mean - std_no_mean, color="C0", alpha=0.3
)
ax1.plot(X_grid, vmapped_mean_function(params, X_grid), label="mean model")
ax1.set_xlabel("x")

plt.legend()
plt.show()